# Linear Regression: Univariate

---
This script contains examples on how to train and test a linear regression model in R. For executing the script, you will need to download the dataset "Admission_Predict.csv".

**Note on importing libraries:**

General syntax to import specific functions in a library:
*library(library_name)*
*library(dplyr)*

**Libraries:**

**dplyr** -- is a grammar of data manipulation, providing a consistent set of verbs that help you solve the most common data manipulation challenges (filtering, selecting, mutating, summarizing, arranging).

**ggplot2** -- is a system for declaratively creating graphics, based on The Grammar of Graphics. You provide the data, tell ggplot2 how to map variables to aesthetics, what graphical primitives to use, and it takes care of the details.

**readr** -- provides a fast and friendly way to read rectangular data (like csv, tsv, and fwf).

**caret** -- (short for Classification And REgression Training) is a set of functions that attempt to streamline the process for creating predictive models.

**corrplot** -- provides a visual exploratory tool on correlation matrix that supports automatic variable reordering to help detect hidden patterns among variables.

**car** - provides a collection of functions for applied regression, linear models, and generalized linear models.

**lmtest** - provides a collection of tests, data sets, and examples for diagnostic checking in linear regression models.

**ggfortify** - provides unified plotting functions for statistics commonly used, such as GLM, time series, PCA families, clustering and survival analysis.

In [ ]:
# Function to install packages if they are not already installed
install_if_missing <- function(packages) {
  for (pkg in packages) {
    if (!require(pkg, character.only = TRUE, quietly = TRUE)) {
      cat("Installing package:", pkg, "\n")
      install.packages(pkg, dependencies = TRUE, repos = "https://cran.rstudio.com/")
      library(pkg, character.only = TRUE)
    }
  }
}

# List of required packages
required_packages <- c(
  "dplyr",       # Data manipulation
  "ggplot2",     # Data visualization
  "readr",       # Reading CSV files
  "caret",       # Machine learning framework
  "corrplot",    # Correlation plots
  "gridExtra",   # Arranging plots
  "broom",       # Tidy model outputs
  "car",         # Regression diagnostics
  "lmtest",      # Linear model tests
  "ggfortify",   # Fortify for ggplot2
  "tidyr",       # Data tidying
  "glmnet"       # Ridge and Lasso regression (for exercises)
)

# Install missing packages and load all libraries
install_if_missing(required_packages)

# Set seed for reproducibility
set.seed(42)

cat("All required packages are installed and loaded successfully!\n")

In [ ]:
# To make this notebook's output stable across runs (we make the output reproducible)
set.seed(42)

In [ ]:
# how to get help on a function
?print

In [ ]:
# Let's generate some linear looking data:
# Note: runif generates samples from the uniform distribution, while rnorm from normal
X <- 2 * runif(100, 0, 1)

In [ ]:
y <- 4 + 3 * X + rnorm(100, 0, 1) # notice a difference between the function to generate X and y? The former draws from a uniform distribution and the latter from a normal distribution.

In [ ]:
print(cbind(X, y))  # cbind combines vectors by columns

In [ ]:
# Let's plot (info on the marker and the color --> see ?plot and ?par for details)
plot(X, y, pch=16, col="blue", xlab="x", ylab="y", xlim=c(0, 2), ylim=c(0, 12))

In [ ]:
# Training a linear model
lin_reg <- lm(y ~ X) # create and fit the linear regression
Y_predict <- predict(lin_reg, data.frame(X = X))

In [ ]:
plot(X, y, pch=16, col="blue")
lines(X[order(X)], Y_predict[order(X)], col="red", lwd=2)

In [ ]:
X_new <- c(0.5, 1.75)
y_predict <- predict(lin_reg, data.frame(X = X_new))
y_predict

**Let's try with real data**

---

Read CSV file = banking.csv

In [ ]:
banking_url <- "https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv"
print("Fetching banking.csv from GitHub...")

In [ ]:
dataset <- read_csv(banking_url)

In [ ]:
head(dataset)

In [ ]:
tail(dataset)

In [ ]:
dim(dataset) # Returns the dimensions of the data frame

In [ ]:
str(dataset) # Returns the structure of the data frame

In [ ]:
# Check for NAs
sapply(dataset, function(x) any(is.na(x))) # Check if any column has missing values

In [ ]:
sapply(dataset, function(x) sum(is.na(x))) # Count missing values per column

In [ ]:
# Describe the data
summary(dataset)

print(colnames(dataset))

In [ ]:
# Select the target and independent variables (predict log1p(duration) with pre-call/macroeconomic features)
# Prepare pdays-related features and avoid leakage by excluding 'duration' and 'y' from X
df_banking <- dataset
df_banking$was_previously_contacted <- as.numeric(df_banking$pdays != 999)
df_banking$pdays_clean <- ifelse(df_banking$pdays == 999, NA, df_banking$pdays)
df_banking$pdays_clean[is.na(df_banking$pdays_clean)] <- median(df_banking$pdays_clean, na.rm = TRUE)

feature_cols_cat <- c('marital', 'education', 'housing', 'loan', 'contact', 'poutcome')
feature_cols_num <- c('age', 'previous', 'pdays_clean', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'was_previously_contacted')

# Create dummy variables for categorical features
X_df <- model.matrix(~ . - 1, data = df_banking[, c(feature_cols_cat, feature_cols_num)])
X <- as.matrix(X_df)
y <- log1p(df_banking$duration)

# Choose one feature for visualization (first column)
viz_col <- colnames(X_df)[1]
viz_idx <- 1

# Preserve banking variables to avoid being overwritten later
X_df_banking <- X_df
y_banking <- y
viz_col_banking <- viz_col
viz_idx_banking <- viz_idx

In [ ]:
# Scatter plot
plot(X_df_banking[, viz_col_banking], y_banking, pch=16, 
     xlab=viz_col_banking, ylab='log1p(duration)')

In [ ]:
# Remove non-numeric vars and check the correlations
numeric_cols <- sapply(dataset, is.numeric)
corrmat <- cor(dataset[, numeric_cols], use = "complete.obs")
corrmat

In [ ]:
# Plot correlation heatmap
corrplot(corrmat, method = "color", type = "upper", order = "hclust", 
         tl.cex = 0.8, tl.col = "black", tl.srt = 45)

In [ ]:
# Split train and test set
set.seed(0) # For reproducibility
train_indices <- createDataPartition(y, p = 0.8, list = FALSE)
X_train <- X[train_indices, ]
X_test <- X[-train_indices, ]
y_train <- y[train_indices]
y_test <- y[-train_indices]

print(dim(X_train))
print(dim(X_test))
print(length(y_train))
print(length(y_test))

# Preserve banking split to avoid being overwritten by later BTC section
X_train_banking <- X_train
X_test_banking <- X_test
y_train_banking <- y_train
y_test_banking <- y_test

In [ ]:
# Fit the model on training set
model <- lm(y_train ~ X_train) # training the algorithm

In [ ]:
# Get coefficients
print(paste('Intercept:', coef(model)[1]))
print(paste('First coefficient:', coef(model)[2]))
print(paste('\nThe fitted model intercept is:', round(coef(model)[1], 2)))

In [ ]:
# Get fitted value on test set
y_test_predicted <- predict(model, data.frame(X_train = X_test))
y_test_predicted_banking <- y_test_predicted

# Compare predictions
comparison_df <- data.frame(True = y_test, Predicted = y_test_predicted)
head(comparison_df, 10)

In [ ]:
# Plot model
x_plot <- X_test[, viz_idx_banking]
y_true <- y_test
y_pred_flat <- y_test_predicted
order_idx <- order(x_plot)

plot(x_plot, y_true, pch=16, col="gray", xlab=viz_col_banking, ylab="log1p(duration)")
lines(x_plot[order_idx], y_pred_flat[order_idx], col="red", lwd=2)

In [ ]:
# Plot some predicted vs true values
points_to_plot <- 30
x_subset <- X_test[1:points_to_plot, viz_idx_banking]
plot(x_subset, y_test[1:points_to_plot], pch=1, col="blue", 
     xlab=viz_col_banking, ylab="Values", main="True vs Predicted")
points(x_subset, y_test_predicted[1:points_to_plot], pch=4, col="red")
legend("topright", legend=c("true value", "predicted"), 
       pch=c(1, 4), col=c("blue", "red"))

In [ ]:
# Evaluate Root Mean Square Error (RMSE)
RMSE_test <- sqrt(mean((y_test - y_test_predicted)^2))
print(paste('Root Mean Squared Error on test set:', RMSE_test))
print(paste('Mean of log1p(duration) y_test:', mean(y_test)))

In [ ]:
# Evaluate R-squared
R2 <- cor(y_test, y_test_predicted)^2
print(paste('R-squared:', R2))

## Check linear regression assumptions

**Linear regression assumes the following:**

---



1. **linear relationship** between regressor(s) and target
2. little or **no multicollinearity** between regressors
3. **homoscedasticity**, i.e. the variance of the error terms (i.e. residuals) doesn't vary too much for all observations
4. **normal distribution of error terms** (i.e. residuals)
5. no correlation between regressors and residuals or little or **no autocorrelation in residuals** for time series, i.e. correlation between  𝑒𝑡  and  𝑒𝑡−1

In [ ]:
# Checking Assumption 1 - linear relationship between regressors and target
# Scatter plot of Y vs x (using selected visualization feature)

plot(X_df_banking[, viz_col_banking], y_banking, pch=16,
     main='Feature vs log1p(duration)', 
     xlab=viz_col_banking, ylab='log1p(duration)')

*Checking Assumption 2 - little to no multicolinearity between regressors *

There is only 1 regressor in a univeriate regression!

In [ ]:
# Checking Assumption 3 - Homoscedasticity
# Plot the residual and check their "shape"
residuals_test <- y_test_banking - y_test_predicted_banking
plot(1:length(residuals_test), residuals_test, pch=16, col=rgb(0,0,0,0.5),
     xlab="Observation Index", ylab="Residuals", main="Residuals vs Index")

In [ ]:
# Checking Assumption 4 - Normal distribution of residuals
# Check if residual distribution looks like a normal distribution with same mean and variance

resid_mean <- mean(residuals_test)
resid_std <- sd(residuals_test)
normal_distr <- rnorm(length(residuals_test), resid_mean, resid_std)

# Create three plots
par(mfrow=c(1,3))

# Plot 1: Residual distribution
hist(residuals_test, main='Residual distribution', xlab='Residuals', breaks=20, col='lightblue')

# Plot 2: Normal distribution
hist(normal_distr, main='Normal distribution', xlab='Values', breaks=20, col='lightgreen')

# Plot 3: Overlay
hist(residuals_test, main='Comparison', xlab='Values', breaks=20, col=rgb(0,0,1,0.5), 
     density=20, angle=45)
hist(normal_distr, add=TRUE, breaks=20, col=rgb(1,0,0,0.5), density=20, angle=-45)
legend("topright", legend=c("residuals", "normal distribution"), 
       fill=c(rgb(0,0,1,0.5), rgb(1,0,0,0.5)))

par(mfrow=c(1,1))

In [ ]:
# Check QQ-plot, i.e. plotting the quantiles of residual against quantiles of normal distribution

percentile_set <- seq(0, 100, length.out=100)
residual_percentile <- quantile(residuals_test, percentile_set/100)
normal_percentile <- quantile(normal_distr, percentile_set/100)

plot(normal_percentile, residual_percentile, pch=1, col="blue",
     xlab='normal percentiles', ylab='residual percentiles')
# plot bisector
abline(a=0, b=1, lty=2, col="black")

In [ ]:
# Checking Assumption 5 - Independence of residuals
# Check correlation between residuals and each regressor (robust to shapes)
resid_vec <- as.vector(residuals_test)
X_test_banking_df <- as.data.frame(X_test_banking)
names(X_test_banking_df) <- colnames(X_df_banking)

# Calculate correlations
corr_with_resid <- sapply(X_test_banking_df, function(col) {
  if(length(unique(col)) > 1) {
    cor(resid_vec, col, use="complete.obs")
  } else {
    NA
  }
})
print(corr_with_resid)

**Linear regression for forecasting**

---

In this next section, we aim to train a linear model that will predict the Close price of the Bitcoin cryptocurrency.

In [ ]:
# Let's import the dataset including Bitcoin prices
btc_url <- "https://raw.githubusercontent.com/umatter/EDFB/main/data/data_BTC.csv"
print("Fetching data_BTC.csv from GitHub...")

In [ ]:
tryCatch({
  data <- read_csv(btc_url)
}, error = function(e) {
  print(paste("Falling back to alternative dataset due to:", e$message))
  # Create a simple synthetic BTC dataset for demonstration
  dates <- seq(as.Date("2020-01-01"), as.Date("2023-12-31"), by="day")
  n <- length(dates)
  # Simulate BTC-like price movement
  set.seed(123)
  returns <- rnorm(n, 0, 0.03)
  prices <- cumprod(c(7000, 1 + returns))[1:n]
  data <<- data.frame(Date = dates, `BTC-USD.Close` = prices)
  names(data)[2] <<- "BTC-USD.Close"
})

# Standardize columns for downstream code
if('Timestamp' %in% names(data) && 'Close' %in% names(data)) {
  # Convert UNIX ms timestamp to datetime
  data$Date <- as.POSIXct(data$Timestamp/1000, origin="1970-01-01")
  # Ensure chronological order
  data <- data[order(data$Date), ]
  # Rename to expected column names
  names(data)[names(data) == 'Close'] <- 'BTC-USD.Close'
} else if('Date' %in% names(data) && 'BTC-USD.Close' %in% names(data)) {
  # If already in expected format, ensure Date is datetime
  data$Date <- as.Date(data$Date)
}

# Remove timestamp if it exists
data$Timestamp <- NULL

# Ensure strict chronological order
data <- data[order(data$Date), ]

In [ ]:
# Let's check if we imported correctly
head(data)

In [ ]:
# Let's get some summary stats on the prices
summary(data)

In [ ]:
# Let's plot the movement of the BTC Close Price
plot(data$Date, data$`BTC-USD.Close`, type='l', col='blue', 
     xlab='Date', ylab='Price (USD)', main='Bitcoin Price Over Time')

In [ ]:
# We can use a linear model to forecast next-period log return using a few lagged log-returns (stationary target).
# Create lagged return features. This function does not mutate its input data frame.
create_lagged_features <- function(data, lag) {
  df <- data
  df$ret <- c(NA, diff(log(df$`BTC-USD.Close`)))
  for(i in 1:lag) {
    df[[paste0('lag_ret_', i)]] <- c(rep(NA, i), df$ret[1:(nrow(df)-i)])
  }
  df <- df[complete.cases(df), ]
  return(df)
}

In [ ]:
# Create lag features on log returns with 5 lags
data <- create_lagged_features(data, lag=5)

In [ ]:
head(data)

In [ ]:
# Split the data into training and testing sets using a chronological split (avoid leakage)
feature_cols <- paste0('lag_ret_', 1:5)
X_btc <- data[, feature_cols]
y_btc <- data$ret
split_idx <- floor(nrow(data) * 0.8)
X_train_btc <- X_btc[1:split_idx, ]
X_test_btc <- X_btc[(split_idx+1):nrow(X_btc), ]
y_train_btc <- y_btc[1:split_idx]
y_test_btc <- y_btc[(split_idx+1):length(y_btc)]

In [ ]:
# Train a linear regression model
model_btc <- lm(y_train_btc ~ ., data = X_train_btc)

# Print the summary, which includes coefficients and p-values
summary(model_btc)

In [ ]:
# Make predictions on the test set
y_pred_btc <- predict(model_btc, X_test_btc)

# Evaluate the model's performance (with naive baseline: last return)
# Calculate the mean squared error
mse <- mean((y_test_btc - y_pred_btc)^2)

# Calculate Root Mean Squared Error (RMSE)
rmse <- sqrt(mse)

# Naive baseline: predict next return as last observed return (lag 1)
naive_pred <- X_test_btc$lag_ret_1
rmse_naive <- sqrt(mean((y_test_btc - naive_pred)^2))

# Calculate the R²
r2 <- cor(y_test_btc, y_pred_btc)^2

cat(sprintf("RMSE (model): %.6f\n", rmse))
cat(sprintf("RMSE (naive lag-1): %.6f\n", rmse_naive))
cat(sprintf("RMSE ratio (model/naive): %.3f\n", rmse/rmse_naive))
cat(sprintf("R-squared on Test Set: %.3f\n", r2))

In [ ]:
for(i in 1:min(10, length(y_test_btc))) {
  cat(sprintf("True: %.2f, Predicted: %.2f\n", y_test_btc[i], y_pred_btc[i]))
}

In [ ]:
# Plot true vs predicted returns over time
plot(1:length(y_test_btc), y_test_btc, type='l', col='blue', 
     xlab='Time Index', ylab='Returns', main='True vs Predicted Returns')
lines(1:length(y_pred_btc), y_pred_btc, col='red')
legend("topright", legend=c("True returns", "Predicted returns"), 
       col=c("blue", "red"), lty=1)

## Exercises

Now it's time to practice! These exercises will help you understand linear regression concepts better and gain hands-on experience with both synthetic and real data.

### Exercise 1: Understanding Model Coefficients

**Task:** Interpret the coefficients from the banking dataset model and understand their business meaning.

**Instructions:**
1. Extract and display the coefficients from the banking model
2. Identify the top 5 features with the largest positive and negative coefficients
3. Explain what these coefficients mean in business terms (how they affect call duration)

In [ ]:
# Exercise 1: Your code here
# Hint: Use the trained model from the banking dataset section
# Create a data frame with feature names and coefficients for easy interpretation

# Your solution:


### Exercise 2: Residual Analysis

**Task:** Perform a comprehensive residual analysis to check model assumptions.

**Instructions:**
1. Calculate residuals for both training and test sets
2. Create a residual vs fitted values plot
3. Test for normality of residuals using a histogram and Q-Q plot
4. Check for patterns that might indicate assumption violations

In [ ]:
# Exercise 2: Your code here
# Hint: Calculate residuals = actual - predicted
# Use par(mfrow=c(2,2)) to create multiple diagnostic plots

# Your solution:


### Exercise 3: Feature Engineering

**Task:** Create new features and see if they improve model performance.

**Business Context:** Sometimes combining existing features or creating polynomial terms can capture non-linear relationships.

**Instructions:**
1. Create interaction terms between two numerical features
2. Add polynomial features (squared terms) for numerical variables
3. Train a new model with these engineered features
4. Compare R² and RMSE with the original model

In [ ]:
# Exercise 3: Your code here
# Hint: Use I(variable^2) for polynomial terms in R formulas
# Be careful about overfitting with too many features

# Your solution:


### Exercise 4: Cross-Validation

**Task:** Use cross-validation to get a more robust estimate of model performance.

**Business Context:** Before deploying a model, you want to ensure it performs consistently across different data samples.

**Instructions:**
1. Use 5-fold cross-validation on the banking dataset
2. Calculate mean and standard deviation of R² scores
3. Compare with a simple baseline model (predicting the mean)
4. Discuss the stability of your model

In [ ]:
# Exercise 4: Your code here
# Hint: Use trainControl and train from caret package

# Your solution:


### Exercise 5: Synthetic Data Generation

**Task:** Create your own synthetic dataset with known relationships and test your model.

**Instructions:**
1. Generate a dataset with 500 observations and 3 features
2. Create a known linear relationship: y = 2*x1 + 3*x2 - 1.5*x3 + noise
3. Add some outliers to make it more realistic
4. Train a linear model and see how well it recovers the true coefficients
5. Experiment with different noise levels

In [ ]:
# Exercise 5: Your code here
# Hint: Use rnorm and runif functions to generate features and noise
# True coefficients should be [2, 3, -1.5]

set.seed(42)  # For reproducibility

# Your solution:


### Exercise 6: Time Series Forecasting Analysis

**Task:** Analyze the Bitcoin forecasting model more deeply.

**Instructions:**
1. Calculate the directional accuracy (how often the model predicts the correct sign of returns)
2. Create a cumulative returns plot comparing actual vs predicted strategies
3. Test different lag lengths (1, 3, 5, 10) and compare performance
4. Discuss the practical implications for trading

In [ ]:
# Exercise 6: Your code here
# Hint: Directional accuracy = percentage of times sign(predicted) == sign(actual)
# Cumulative returns = cumsum of returns over time

# Your solution:


### Exercise 7: Model Comparison

**Task:** Compare linear regression with other simple models.

**Instructions:**
1. Implement a Ridge regression model on the banking dataset
2. Implement a Lasso regression model
3. Compare performance (R², RMSE) of all three models
4. Discuss when you might prefer each approach

In [ ]:
# Exercise 7: Your code here
# Hint: Use glmnet package for Ridge and Lasso regression
# You may need to tune the alpha parameter
# Note: glmnet is already loaded in the first cell

# Your solution:

### Exercise 8: Business Impact Analysis

**Task:** Quantify the business value of your duration prediction model.

**Business Context:** Longer calls might indicate higher customer engagement and conversion probability.

**Scenario:**
- Calls < 2 minutes (log duration < 0.69): Low engagement
- Calls 2-5 minutes (log duration 0.69-1.61): Medium engagement  
- Calls > 5 minutes (log duration > 1.61): High engagement
- You want to prioritize follow-up with high engagement predictions

**Instructions:**
1. Classify actual and predicted durations into engagement categories
2. Calculate accuracy for each engagement level
3. Estimate the business value of correctly identifying high-engagement calls

In [ ]:
# Exercise 8: Your code here
# Hint: Use cut() function to create engagement categories
# Calculate confusion matrix for the three categories

# Your solution:


### Exercise 9: Advanced Diagnostics

**Task:** Perform advanced model diagnostics to identify potential issues.

**Instructions:**
1. Calculate Cook's distance to identify influential observations
2. Check for multicollinearity using Variance Inflation Factor (VIF)
3. Perform the Durbin-Watson test for autocorrelation in residuals
4. Suggest improvements based on your findings

In [ ]:
# Exercise 9: Your code here
# Hint: Use cooks.distance(), vif() from car package, and dwtest() from lmtest
# Cook's distance identifies outliers that heavily influence the model

# Your solution:


### Reflection Questions

After completing the exercises, consider these questions:

1. **When might linear regression not be appropriate?**
   - Think about non-linear relationships, categorical outcomes, etc.

2. **How do you balance model complexity with interpretability?**
   - Consider the trade-off between adding features and keeping the model simple

3. **What are the key assumptions of linear regression and why do they matter?**
   - Relate each assumption to potential business consequences if violated

4. **How would you explain R² to a non-technical business stakeholder?**
   - Focus on practical interpretation rather than mathematical formula

5. **In what business scenarios would you prefer RMSE over R² as an evaluation metric?**
   - Think about when absolute prediction errors matter more than explained variance

6. **How might you improve the Bitcoin forecasting model?**
   - Consider external factors, different features, or alternative approaches

### Additional Challenges

For further learning:
- Try implementing linear regression from scratch using matrix operations
- Explore regularization techniques (Ridge, Lasso, Elastic Net) in more detail
- Practice with different datasets (housing prices, stock returns, sales forecasting)
- Learn about advanced time series models (ARIMA, GARCH) for financial data
- Investigate non-linear regression techniques (polynomial, spline regression)

### Key Takeaways

- Linear regression is a powerful but simple tool for understanding relationships
- Always check model assumptions before trusting predictions
- Feature engineering can significantly improve model performance
- Cross-validation provides more reliable performance estimates
- Business context should guide model evaluation and interpretation
- Simple models are often preferable for interpretability and robustness